In [2]:
import pandas as pd
import numpy as np

PROJECT_ROOT="/content/drive/MyDrive/behavior-aware-bms"

battery=pd.read_csv(
    f"{PROJECT_ROOT}/data/features/battery_rul_estimation_v1.csv"
)

print(battery.shape)
battery.head()

(34, 22)


,battery_id,avg_stress,avg_temp,fast_charge_duration,deep_discharge_duration,high_temp_duration,aggressive_discharge_count,avg_soc,aging_budget,stress_norm,...,fc_norm,soc_norm,health_index,battery_state,remaining_health,consumed_life,equivalent_aging_factor,estimated_total_cycles,rul_cycles,replacement_policy
0,B0005,13.670454,26.369701,1.250087e+09,1.277054e+07,3581775.549,45284,NaN,35,0.136705,...,1.000000,0,65,DEGRADED,35,65,0.545425,1833.432609,642,NORMAL
1,B0006,11.407560,26.429154,1.027942e+09,1.931800e+07,4680506.022,44512,NaN,35,0.114076,...,0.822297,0,65,DEGRADED,35,65,0.520900,1919.752623,672,NORMAL
2,B0007,0.065179,26.119363,6.213354e+05,1.428340e+07,4420739.468,195,NaN,42,0.000652,...,0.000497,0,58,WARNING,42,58,0.366672,2727.236292,1145,NORMAL
3,B0018,14.449429,25.913199,6.962533e+08,1.054231e+07,0.000,32084,NaN,35,0.144494,...,0.556964,0,65,DEGRADED,35,65,0.476063,2100.562114,735,NORMAL
4,B0025,16.337941,28.482356,2.693701e+08,1.155482e+07,2827773.483,4349,NaN,35,0.163379,...,0.215481,0,65,DEGRADED,35,65,0.437970,2283.261517,799,NORMAL


In [3]:
print(battery.columns.tolist())

['battery_id', 'avg_stress', 'avg_temp', 'fast_charge_duration', 'deep_discharge_duration', 'high_temp_duration', 'aggressive_discharge_count', 'avg_soc', 'aging_budget', 'stress_norm', 'temp_norm', 'dd_norm', 'fc_norm', 'soc_norm', 'health_index', 'battery_state', 'remaining_health', 'consumed_life', 'equivalent_aging_factor', 'estimated_total_cycles', 'rul_cycles', 'replacement_policy']


In [4]:
def generate_explanation(row):

    causes=[]

    if row["avg_temp"]>35:
        causes.append(
            "high temperature exposure"
        )

    if row["fast_charge_duration"]>50:
        causes.append(
            "frequent fast charging"
        )

    if row["deep_discharge_duration"]>50:
        causes.append(
            "deep discharge events"
        )

    if row["health_index"]>60:
        causes.append(
            "accelerated battery aging"
        )

    if len(causes)==0:
        causes.append(
            "normal battery usage"
        )

    return ", ".join(causes)

In [5]:
battery["primary_causes"] = (
    battery
    .apply(
        generate_explanation,
        axis=1
    )
)

In [6]:
def recommendation(row):

    if row["battery_state"]=="CRITICAL":
        return (
            "Immediate inspection and battery replacement"
        )

    elif row["battery_state"]=="DEGRADED":
        return (
            "Reduce fast charging and monitor temperature"
        )

    elif row["battery_state"]=="WARNING":
        return (
            "Maintain SOC between 20 and 80 percent"
        )

    return (
        "Continue normal operation"
    )

In [7]:
battery["recommendation"] = (
    battery
    .apply(
        recommendation,
        axis=1
    )
)

In [8]:
battery["guardian_report"] = (

    "Battery "

    + battery["battery_id"].astype(str)

    + " is in "

    + battery["battery_state"]

    + " state with estimated remaining life of "

    + battery["rul_cycles"].astype(int).astype(str)

    + " cycles. Primary degradation factors include "

    + battery["primary_causes"]

    + ". Recommended action: "

    + battery["recommendation"]
)

In [9]:
battery[
    [
        "battery_id",
        "battery_state",
        "rul_cycles",
        "guardian_report"
    ]
].head(10)

,battery_id,battery_state,rul_cycles,guardian_report
0,B0005,DEGRADED,642,Battery B0005 is in DEGRADED state with estima...
1,B0006,DEGRADED,672,Battery B0006 is in DEGRADED state with estima...
2,B0007,WARNING,1145,Battery B0007 is in WARNING state with estimat...
3,B0018,DEGRADED,735,Battery B0018 is in DEGRADED state with estima...
4,B0025,DEGRADED,799,Battery B0025 is in DEGRADED state with estima...
5,B0026,DEGRADED,802,Battery B0026 is in DEGRADED state with estima...
6,B0027,WARNING,1346,Battery B0027 is in WARNING state with estimat...
7,B0028,DEGRADED,863,Battery B0028 is in DEGRADED state with estima...
8,B0029,CRITICAL,245,Battery B0029 is in CRITICAL state with estima...
9,B0030,CRITICAL,246,Battery B0030 is in CRITICAL state with estima...


In [10]:
dashboard = battery[
    [
        "battery_id",
        "health_index",
        "remaining_health",
        "battery_state",
        "rul_cycles",
        "replacement_policy"
    ]
]

dashboard.head()

,battery_id,health_index,remaining_health,battery_state,rul_cycles,replacement_policy
0,B0005,65,35,DEGRADED,642,NORMAL
1,B0006,65,35,DEGRADED,672,NORMAL
2,B0007,58,42,WARNING,1145,NORMAL
3,B0018,65,35,DEGRADED,735,NORMAL
4,B0025,65,35,DEGRADED,799,NORMAL


In [11]:
state_dist = (
    battery["battery_state"]
    .value_counts()
)

print(state_dist)

battery_state
DEGRADED    16
WARNING     13
CRITICAL     5
Name: count, dtype: int64


In [12]:
battery.to_csv(
    f"{PROJECT_ROOT}/data/features/battery_guardian_output_v1.csv",
    index=False
)

In [13]:
state_dist.to_csv(
    f"{PROJECT_ROOT}/reports/metrics/battery_state_distribution.csv"
)

In [14]:
doc="""
# Battery Guardian AI

Inputs:
- Battery behavior features
- Risk engine
- Health index
- Remaining useful life

Outputs:
- Battery state
- Degradation causes
- Replacement policy
- Human-readable explanation

Suitable for:
- Explainable AI
- Industrial BMS
- Digital Twin BMS
- Cloud Battery Management Systems
"""

with open(
    f"{PROJECT_ROOT}/docs/battery_guardian.md",
    "w"
) as f:
    f.write(doc)

print("saved")

saved


In [15]:
print(state_dist)

battery_state
DEGRADED    16
WARNING     13
CRITICAL     5
Name: count, dtype: int64


In [16]:
print(
    battery["guardian_report"].iloc[0]
)

Battery B0005 is in DEGRADED state with estimated remaining life of 642 cycles. Primary degradation factors include frequent fast charging, deep discharge events, accelerated battery aging. Recommended action: Reduce fast charging and monitor temperature


In [17]:
def severity_message(state):

    if state=="CRITICAL":
        return "High risk of battery failure"

    elif state=="DEGRADED":
        return "Battery performance degradation detected"

    elif state=="WARNING":
        return "Early degradation indicators detected"

    return "Battery operating normally"

In [18]:
battery["guardian_status"] = (
    battery["battery_state"]
    .apply(severity_message)
)

In [19]:
battery["guardian_report"] = (

    "Battery "
    + battery["battery_id"].astype(str)

    + ": "

    + battery["guardian_status"]

    + ". Current state: "

    + battery["battery_state"]

    + ". Estimated remaining life: "

    + battery["rul_cycles"].astype(int).astype(str)

    + " cycles. Main degradation contributors: "

    + battery["primary_causes"]

    + ". Recommended action: "

    + battery["recommendation"]
)

In [20]:
battery.to_csv(
    f"{PROJECT_ROOT}/data/features/battery_guardian_output_v1.csv",
    index=False
)


In [21]:
state_dist.to_csv(
    f"{PROJECT_ROOT}/reports/metrics/battery_state_distribution.csv"
)